run this headless

conda activate guitarmidi
screen  jupyter nbconvert --to notebook --execute traning.ipynb --output=training_out.ipynb --ExecutePreprocessor.timeout=-1 > nbconvert.log 2>&1 &


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback,ReduceLROnPlateau
from model import build_cnn_model # Assumes build_cnn_model is adapted for single output and no explicit name
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
import glob # To list files
import time
from IPython.display import clear_output
from datetime import datetime # Import datetime
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES # Removed OUTPUT_DIM_ONSETS

# --- Essential for GPU memory management ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision policy set to 'mixed_float16'.")

        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # Corrected
        print("Memory growth enabled for GPUs.")
    except RuntimeError as e:
        print(f"Error configuring GPU: {e}")
# ---------------------------------------------------------------------------------

print(f"TensorFlow version: {tf.__version__}")

# --- Configuration (using values from serialization part) ---

LEARNING_RATE = 0.001
BATCH_SIZE = 16 # Adjust as needed
EPOCHS = 100

# Directories where slices were saved
input_data_dir = '../data_slices/input'
output_data_dir = '../data_slices/output' # This will be for the 'note' labels
# Removed: onsets_data_dir = 'data_slices/onsets' # No longer needed

# --- Custom Callback for Live Loss Plotting (ADAPTED FOR UNNAMED OUTPUT) ---
class JupyterLivePlottingCallback(Callback):
    def __init__(self, fig_title="Training Metrics", base_plot_dir="training_plots"):
        super().__init__()
        self.fig_title = fig_title
        
        # Generate a timestamp for the directory name
        timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        self.plot_save_dir = os.path.join(base_plot_dir, f"run_{timestamp}")
        
        # Create the directory if it doesn't exist
        os.makedirs(self.plot_save_dir, exist_ok=True)
        print(f"Plots will be saved to: {self.plot_save_dir}")

        # Changed keys from 'note_output_loss' to 'loss' and 'note_output_accuracy' to 'accuracy'
        self.epoch_data = {
            'loss': [], 'accuracy': [], # Total loss/accuracy (since it's the only output)
            'val_loss': [], 'val_accuracy': []
        }
        self.epochs = []

    def on_train_begin(self, logs=None):
        self.epoch_data = {k: [] for k in self.epoch_data.keys()}
        self.epochs = []
        print("Starting Keras model training with live plot. Output will update below...")

    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)

        self.epochs.append(epoch + 1)
        # Fetching general 'loss', 'accuracy', 'val_loss', 'val_accuracy'
        self.epoch_data['loss'].append(logs.get('loss'))
        self.epoch_data['accuracy'].append(logs.get('accuracy'))
        self.epoch_data['val_loss'].append(logs.get('val_loss'))
        self.epoch_data['val_accuracy'].append(logs.get('val_accuracy'))

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle(self.fig_title)

        # Plot Accuracy (now generic)
        axes[0].plot(self.epochs, self.epoch_data['accuracy'], 'b-o', label='Training Accuracy')
        axes[0].plot(self.epochs, self.epoch_data['val_accuracy'], 'r-x', label='Validation Accuracy')
        axes[0].set_title('Accuracy') # Title changed
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].grid(True)
        axes[0].legend(loc='lower right')
        axes[0].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])
        
        # Plot Total Loss (already generic)
        axes[1].plot(self.epochs, self.epoch_data['loss'], 'b-o', label='Total Training Loss')
        axes[1].plot(self.epochs, self.epoch_data['val_loss'], 'r-x', label='Total Validation Loss')
        axes[1].set_title('Total Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].grid(True)
        axes[1].legend(loc='upper right')
        axes[1].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        # Save the plot to a file inside the timestamped directory
        plot_filename = os.path.join(self.plot_save_dir, f"training_plot.png")
        plt.savefig(plot_filename, dpi=150)
        plt.close(fig) # Close the figure to free up memory

        plt.show()

    def on_train_end(self, logs=None):
        print("Training finished. Final plot above.")


# --- 2. Compile the Model (Updated for single, unnamed output) ---
cnn_model = build_cnn_model(INPUT_SHAPE, OUTPUT_DIM_NOTES) 

# Define loss and metrics without a dictionary, as there's only one output
# Keras will automatically apply 'binary_crossentropy' as the loss and 'accuracy' as the metric
# to the single output.
cnn_model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE,clipnorm=1.0),
                  loss='binary_crossentropy', # Changed from dictionary to direct string
                  metrics=['accuracy'])      # Changed from dictionary to list of strings

cnn_model.summary()

# --- 3. Data Loading and Preparation (No changes needed here from previous single-output script) ---

# Get lists of all input and output file paths
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '*.npy')))
output_filepaths = sorted(glob.glob(os.path.join(output_data_dir, '*.npy'))) # For notes

total_samples_on_disk = len(input_filepaths)
if total_samples_on_disk == 0:
    print(f"ERROR: No .npy files found in {input_data_dir}. Please run the serialization script first.")
    exit()
if total_samples_on_disk != len(output_filepaths):
    print("ERROR: Mismatch in number of input or note output files.")
    exit()

print(f"Found {total_samples_on_disk} files on disk.")

# Function to load a single image and its labels from file paths
def load_input_from_files(input_path_tensor):
    input_path = input_path_tensor.numpy().decode('utf-8')
    image = np.load(input_path).astype(np.float32).reshape(INPUT_SHAPE)
    image = tf.ensure_shape(image, INPUT_SHAPE)
    return image

def load_notes_from_files(note_output_path_tensor):
    note_output_path = note_output_path_tensor.numpy().decode('utf-8')
    note_label = np.load(note_output_path).astype(np.float32)
    note_label = np.clip(note_label, 0.0, 1.0) 
    note_label = note_label.reshape(OUTPUT_DIM_NOTES)
    note_label = tf.ensure_shape(note_label, (OUTPUT_DIM_NOTES,))
    return note_label

# TensorFlow wrapper function
def tf_load_sample_from_files(ipath, nopath):
    image = tf.py_function(
        load_input_from_files, [ipath], [tf.float32]
    )[0]
    note_label = tf.py_function(
        load_notes_from_files, [nopath], [tf.float32]
    )[0]
    
    image.set_shape(INPUT_SHAPE)
    note_label.set_shape((OUTPUT_DIM_NOTES,))
    
    return (image, note_label)

# Create a dataset from the lists of file paths
dataset = tf.data.Dataset.from_tensor_slices((input_filepaths, output_filepaths))

# Shuffle the list of paths first
dataset = dataset.shuffle(buffer_size=total_samples_on_disk)

# Split the dataset into training and validation subsets based on indices
split_ratio = 0.7
num_train = int(total_samples_on_disk * split_ratio)

train_dataset = dataset.take(num_train)
val_dataset = dataset.skip(num_train)

# Map the loading function to the datasets
train_dataset = train_dataset.map(
    tf_load_sample_from_files,
    num_parallel_calls=tf.data.AUTOTUNE
)
val_dataset = val_dataset.map(
    tf_load_sample_from_files,
    num_parallel_calls=tf.data.AUTOTUNE
)

# Apply batching and prefetching
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- 3. Configure Callbacks (monitor names updated to generic accuracy/loss) ---
early_stopping_acc = EarlyStopping(
    monitor='val_accuracy', # Changed to generic 'val_accuracy'
    patience=10,
    mode='max',
    verbose=1,
    restore_best_weights=True
)

model_checkpoint_acc = ModelCheckpoint(
    'best_model_by_accuracy.keras', # Filename changed
    monitor='val_accuracy', # Changed to generic 'val_accuracy'
    mode='max',
    save_best_only=True,
    verbose=1
)
reduce_lr_on_plateau = ReduceLROnPlateau(
    monitor='val_accuracy', # Changed to generic 'val_accuracy'
    factor=0.5,
    patience=5,
    mode='max',
    min_delta=0.0001,
    cooldown=0,
    min_lr=0.00001,
    verbose=1
)
jupyter_live_plot = JupyterLivePlottingCallback(fig_title="Keras Training Progress (Single-Output)")

# --- 4. Training the Model ---
print("\n--- Training Pure CNN Model (Single-Output, Unnamed Output) ---")
try:
    history_cnn = cnn_model.fit(train_dataset,
                                epochs=EPOCHS,
                                validation_data=val_dataset,
                                callbacks=[model_checkpoint_acc, early_stopping_acc, jupyter_live_plot,reduce_lr_on_plateau])
    cnn_model.save_weights('guitarmidi-single-unnamed-output-final.weights.h5')
    cnn_model.save('guitarmidi-unnamed.keras')
    print("Final model weights saved successfully!")
except Exception as e:
    print(f"An error occurred during training: {e}")

Mixed precision policy set to 'mixed_float16'.
Memory growth enabled for GPUs.
TensorFlow version: 2.19.0


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 256, 312, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256, 312, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 128, 156, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 156, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128, 156, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 78, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 78, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64, 78, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 39, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ note_output (Dense)             │ (None, 89)             │        22,873 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 394,329 (1.50 MB)

 Trainable params: 393,433 (1.50 MB)

 Non-trainable params: 896 (3.50 KB)

Found 49125 files on disk.
Plots will be saved to: training_plots/run_20250708-000502

--- Training Pure CNN Model (Single-Output) ---
Starting Keras model training with live plot. Output will update below...
Epoch 1/100
An error occurred during training: 'NoneType' object is not iterable


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history_cnn.history['loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Training Metrics")

# Note Accuracy
axes[0, 0].plot(epochs, history_cnn.history['note_output_accuracy'], 'b-o', label='Training Note Accuracy')
axes[0, 0].plot(epochs, history_cnn.history['val_note_output_accuracy'], 'r-x', label='Validation Note Accuracy')
axes[0, 0].set_title('Note Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].grid(True)
axes[0, 0].legend(loc='lower right')

# Onsets Accuracy
axes[0, 1].plot(epochs, history_cnn.history['onsets_output_accuracy'], 'b-o', label='Training Onsets Accuracy')
axes[0, 1].plot(epochs, history_cnn.history['val_onsets_output_accuracy'], 'r-x', label='Validation Onsets Accuracy')
axes[0, 1].set_title('Onsets Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True)
axes[0, 1].legend(loc='lower right')

# Total Loss
axes[1, 0].plot(epochs, history_cnn.history['loss'], 'b-o', label='Total Training Loss')
axes[1, 0].plot(epochs, history_cnn.history['val_loss'], 'r-x', label='Total Validation Loss')
axes[1, 0].set_title('Total Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True)
axes[1, 0].legend(loc='upper right')

# Individual Losses
axes[1, 1].plot(epochs, history_cnn.history['note_output_loss'], 'g-o', label='Training Note Loss')
axes[1, 1].plot(epochs, history_cnn.history['val_note_output_loss'], 'g--x', label='Validation Note Loss')
axes[1, 1].plot(epochs, history_cnn.history['onsets_output_loss'], 'm-o', label='Training Onsets Loss')
axes[1, 1].plot(epochs, history_cnn.history['val_onsets_output_loss'], 'm--x', label='Validation Onsets Loss')
axes[1, 1].set_title('Individual Task Losses')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True)
axes[1, 1].legend(loc='upper right')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
cnn_model.save('guitarmidi.keras')